In [ ]:
from cls import *
import matplotlib.pyplot as plt
from scipy.special import expit
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

In [ ]:
qs = np.arange(0.1, 0.9, 0.1)
Ts = np.array([180, 90, 60, 50, 40, 30, 30, 20])
arms_grid = np.array([2, 4, 6, 8])

gams = [0.5, 4.0, 4.0, 4.0, 0.5, 4.0, 4.0, 4.0]
xs = [2.0, 0.5, 1.0, 1.5, 1.0, -0.5, -1.0, -1.5]
f = expit

In [ ]:
def load_gittins_interp(filename):
    data = np.load(filename, allow_pickle=True)
    xs = data["xs"]
    ys = data["ys"]
    kind = str(data["kind"])
    Interp = {"pchip": PchipInterpolator, "cubic": CubicSpline}[kind]
    return Interp(xs, ys, extrapolate=True)

In [ ]:
def _one_sample(seed, T, q, arms, lbda):
    np.random.seed(int(seed))
    Xs = []
    for i in range(arms):
        interp = load_gittins_interp('interp/{:.1f}-{:.1f}-{:.2f}-{:.1f}.npz'.format(q, lbda, gams[i], xs[i]))
        Xs.append(OU(xs[i], gams[i], f=f, T = T, lbda=lbda, interp=interp, q=q))
    return ComputePathwise(Xs, T, q, myopic=True)

def run(N, T, q, arms, lbda, n_jobs):
    seeds = np.random.SeedSequence().generate_state(N)
    out = Parallel(n_jobs=n_jobs, backend="loky", verbose=5)(
        delayed(_one_sample)(int(s), T, q, arms, lbda=lbda) for s in seeds
    )
    g, b = map(np.array, zip(*out))
    return g, b

In [ ]:
def sweep(qs, arms_grid, N, Ts, lbda, n_jobs=-1):
    mean_gap = np.empty((len(qs), len(arms_grid)))
    se_gap = np.empty_like(mean_gap)
    for i, q in enumerate(qs):
        for j, arms in enumerate(arms_grid):
            g, b = run(N=N, T=Ts[i], q=float(q), arms=int(arms), lbda=lbda, n_jobs=n_jobs)
            d = g - b
            mean_gap[i, j] = d.mean()
            se_gap[i, j] = d.std(ddof=1) / np.sqrt(len(d))
    return mean_gap, se_gap

In [ ]:
lbdas = [0.5, 1, 1.5]
results = [sweep(qs, arms_grid, N=5000, lbda=lb, Ts=Ts) for lb in lbdas]

In [ ]:
vmax = max(np.nanmax(m) for m, _ in results)
norm = PowerNorm(gamma=0.4, vmin=0.0, vmax=vmax)
cmap = "Reds"

fig, axes = plt.subplots(1, len(lbdas), figsize=(15, 4.6),
                         sharey=True, constrained_layout=True)

for ax, lb, (mean_gap, se_gap) in zip(axes, lbdas, results):
    t_gap = mean_gap / se_gap
    im = ax.imshow(mean_gap.T, origin="lower", aspect="auto", cmap=cmap, norm=norm)
    ax.set_xticks(range(len(qs)), labels=[f"{q:.1f}" for q in qs])
    ax.set_xlabel(r"$q$")
    ax.set_title(rf"$\lambda = {lb}$")
    for i in range(len(qs)):
        for j in range(len(arms_grid)):
            sig   = abs(t_gap[i, j]) > 2
            shade = norm(mean_gap[i, j])
            tcol  = ("white" if shade > 0.6 else "black") if sig else "0.6"
            ax.text(i, j, f"{mean_gap[i, j]:.1e}\n$t={t_gap[i, j]:.1f}$",
                    ha="center", va="center", fontsize=5.5, color=tcol)

axes[0].set_yticks(range(len(arms_grid)), labels=[str(a) for a in arms_grid])
axes[0].set_ylabel("number of arms")

fig.colorbar(im, ax=axes, label=r"$\mathbb{E}[\mathrm{GI}-\mathrm{M.o}]$", shrink=0.9)
fig.suptitle(r"$\mathbb{E}[\mathrm{GI}-\mathrm{M.o}]$ by $\lambda$")
plt.show()